In [0]:
# Configuration

CATALOG = "worldbank_ai"
BRONZE_SCHEMA = "bronze"

RAW_DOCUMENTS_VOLUME = "raw_documents"

print(f"Catalog: {CATALOG}")
print(f"Schema: {BRONZE_SCHEMA}")
print(f"Volume: {RAW_DOCUMENTS_VOLUME}")

Catalog: worldbank_ai
Schema: bronze
Volume: raw_documents


In [0]:
# Create the volume
spark.sql(f"""
CREATE VOLUME IF NOT EXISTS
    {CATALOG}.{BRONZE_SCHEMA}.{RAW_DOCUMENTS_VOLUME}
COMMENT 'Raw World Bank Global Economic Prospects PDF documents'
""")

print(
    f"Volume ready: "
    f"{CATALOG}.{BRONZE_SCHEMA}.{RAW_DOCUMENTS_VOLUME}"
)

Volume ready: worldbank_ai.bronze.raw_documents


In [0]:
# Verify the volume

volumes_df = spark.sql(
    f"SHOW VOLUMES IN {CATALOG}.{BRONZE_SCHEMA}"
)

display(volumes_df)

database,volume_name
bronze,raw_documents


In [0]:
# Define the document path

VOLUME_PATH = (
    f"/Volumes/{CATALOG}/"
    f"{BRONZE_SCHEMA}/"
    f"{RAW_DOCUMENTS_VOLUME}"
)

GEP_PATH = f"{VOLUME_PATH}/global_economic_prospects"

print("Volume path:")
print(VOLUME_PATH)

print("\nGEP document path:")
print(GEP_PATH)

Volume path:
/Volumes/worldbank_ai/bronze/raw_documents

GEP document path:
/Volumes/worldbank_ai/bronze/raw_documents/global_economic_prospects


In [0]:
# Create the folder
dbutils.fs.mkdirs(GEP_PATH)

print(f"Directory ready: {GEP_PATH}")

Directory ready: /Volumes/worldbank_ai/bronze/raw_documents/global_economic_prospects


In [0]:
# Check the folder
files = dbutils.fs.ls(GEP_PATH)

if not files:
    print("GEP directory exists and is currently empty.")
else:
    for file in files:
        print(
            file.name,
            file.size,
            file.path
        )

GEP directory exists and is currently empty.


In [0]:
EXPECTED_FILES = {
    "GEP-Jan-2022.pdf",
    "GEP-Jan-2023.pdf",
    "GEP-Jan-2024.pdf",
    "GEP-Jan-2025.pdf",
    "GEP-Jan-2026.pdf"
}

actual_files = {
    file.name
    for file in dbutils.fs.ls(GEP_PATH)
    if not file.isDir()
}

missing_files = EXPECTED_FILES - actual_files
unexpected_files = actual_files - EXPECTED_FILES

print(f"Expected documents: {len(EXPECTED_FILES)}")
print(f"Documents found: {len(actual_files)}")

if missing_files:
    print("\nMissing documents:")
    for file in sorted(missing_files):
        print(f"  - {file}")

if unexpected_files:
    print("\nUnexpected documents:")
    for file in sorted(unexpected_files):
        print(f"  - {file}")

if not missing_files:
    print("\nAll expected GEP documents are present.")

Expected documents: 5
Documents found: 5

All expected GEP documents are present.


In [0]:
# Validate that they're not empty files


MINIMUM_PDF_SIZE_BYTES = 100_000

validation_results = []

for file in dbutils.fs.ls(GEP_PATH):

    if file.isDir():
        continue

    validation_results.append({
        "file_name": file.name,
        "size_bytes": file.size,
        "is_pdf": file.name.lower().endswith(".pdf"),
        "size_valid": file.size >= MINIMUM_PDF_SIZE_BYTES
    })

validation_df = spark.createDataFrame(validation_results)

display(validation_df)

file_name,is_pdf,size_bytes,size_valid
GEP-Jan-2022.pdf,true,4571994,true
GEP-Jan-2023.pdf,true,3951634,true
GEP-Jan-2024.pdf,true,6667286,true
GEP-Jan-2025.pdf,true,4214266,true
GEP-Jan-2026.pdf,true,3647450,true


In [0]:
# Final check

if missing_files:
    raise RuntimeError(
        f"Missing required GEP documents: {sorted(missing_files)}"
    )

invalid_files = [
    row["file_name"]
    for row in validation_results
    if not row["is_pdf"] or not row["size_valid"]
]

if invalid_files:
    raise RuntimeError(
        f"Document validation failed: {invalid_files}"
    )

print("=" * 60)
print("GEP RAW DOCUMENT STORAGE VALIDATION")
print("=" * 60)

print(f"Volume: {CATALOG}.{BRONZE_SCHEMA}.{RAW_DOCUMENTS_VOLUME}")
print(f"Path: {GEP_PATH}")
print(f"Documents: {len(actual_files)}")
print("Years: 2022-2026")
print("Status: READY")

GEP RAW DOCUMENT STORAGE VALIDATION
Volume: worldbank_ai.bronze.raw_documents
Path: /Volumes/worldbank_ai/bronze/raw_documents/global_economic_prospects
Documents: 5
Years: 2022-2026
Status: READY
